# Mervin/Mervis -- Gemma 4 E4B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Gemma 4 E4B-it** on the Mervin/Mervis persona dataset with
[Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit **Q4_K_M GGUF**
(~5.3 GB), and uploads it to Hugging Face -- where `serve.py` auto-downloads it.

**No system prompt.** Training data is bare `user -> assistant`; the model
produces the Mervin/Mervis format purely from fine-tuning. Verified on A100.

### Before you run
- Runtime -> Change runtime type -> **GPU** (A100/L4; T4 works but slower for E4B).
- Add a Colab **secret** `HF_TOKEN` (key icon) with a Hugging Face **write** token.

### Gemma 4 notes (handled below)
- transformers `>4.56.2,<=5.5.0` (the `gemma4` arch is not in 4.56.2); `peft>=0.19.0`.
- Gemma 4's tokenizer is a multimodal *processor* -- inference renders the
  template to text then tokenizes with its inner text tokenizer.
- The GGUF converter also emits a `*-mmproj` vision projector; we upload only
  the text Q4_K_M.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Unsloth + a dependency set it supports. Gemma 4 / newer archs need a newer
# transformers than 4.56.2; bump to the newest Unsloth supports (<=5.5.0).
# peft>=0.19.0 for newer arch LoRA targets. Drop the runtime's broken torchao.
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers>4.56.2,<=5.5.0,!=5.0.0,!=5.1.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft>=0.19.0"
!pip uninstall -q -y torchao
import importlib.metadata as m
print("transformers", m.version("transformers"), "| trl", m.version("trl"),
      "| datasets", m.version("datasets"), "| peft", m.version("peft"))

In [ ]:
import os
# Some Colab torch images ship a broken torch.compile/inductor that crashes
# Unsloth at import (ImportError: cannot import 'CUSTOM_KEY'). Run eager to be safe.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel
MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded gemma-4-E4B-it")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset -- no system prompt

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"

raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

# NO system prompt. Rows are 1-, 2-, or 3-turn (prompt/response, +prompt2/response2,
# +prompt3/response3). Render the WHOLE conversation with the model's chat template
# so it learns to keep the Mervin/Mervis format on FOLLOW-UP turns, not just turn 1.
def to_text(r):
    msgs = [{"role": "user", "content": r["prompt"]},
            {"role": "assistant", "content": r["response"]}]
    for i in ("2", "3"):
        p, a = (r.get("prompt" + i) or "").strip(), (r.get("response" + i) or "").strip()
        if p and a:
            msgs += [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list([to_text(r) for r in rows])
n2 = sum(1 for r in rows if r.get("prompt2"));  n3 = sum(1 for r in rows if r.get("prompt3"))
print(f"{len(rows)-n2} single, {n2-n3} two-turn, {n3} three-turn")
print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 6,   # 3 epochs dropped a tag on follow-up turns -> 6 for a reliable both-tag format
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 10,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Sanity check -- bare user message, no system prompt

In [ ]:
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)  # inner text tokenizer if processor

def ask(q):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for q in ["What is 2+2?", "Tell me about Mondays.", "What is the capital of France?"]:
    print("Q:", q); print(ask(q)); print("-"*60)

In [ ]:
# Both-tags consistency check: every reply must have BOTH <Mervin> and <Mervis>.
# The 3-epoch model dropped one too often, especially on the SECOND turn of a
# conversation -> 6 epochs, and we verify multi-turn here, not just single-turn.
import re

def ask_msgs(msgs):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def both_tags(t):
    return bool(re.search(r"<Mervin>.*?</Mervin>", t, re.S) and re.search(r"<Mervis>.*?</Mervis>", t, re.S))

SINGLE = ["What is 2+2?", "Tell me about Mondays.", "Recommend a book.", "Explain gravity."]
# The real regression: the 2nd reply dropping a tag. Verify follow-up turns too.
PAIRS = [("What do you think about Mondays?", "And what about Fridays?"),
         ("Should I get a cat?", "What about a dog instead?"),
         ("Is it going to rain tomorrow?", "And the day after?"),
         ("Tell me about the ocean.", "What lives in it?")]

ok = tot = 0
for q in SINGLE:
    r = ask_msgs([{"role": "user", "content": q}])
    g = both_tags(r); ok += g; tot += 1
    print(("OK " if g else "BAD"), "T1 |", q, "->", r[:80])
for q1, q2 in PAIRS:
    r1 = ask_msgs([{"role": "user", "content": q1}])
    r2 = ask_msgs([{"role": "user", "content": q1},
                   {"role": "assistant", "content": r1},
                   {"role": "user", "content": q2}])
    g = both_tags(r2); ok += g; tot += 1
    print(("OK " if g else "BAD"), "T2 |", q2, "->", r2[:80])
print(f"\nboth-tags consistent: {ok}/{tot}  (T2 = 2nd-turn replies, the regression we fixed)")

## Export Q4_K_M GGUF + upload to Hugging Face

Uploads the text GGUF to `freeideas/merv-gemma4e4b` as `model-q4_k_m.gguf`.
Token from the Colab `HF_TOKEN` secret.

In [ ]:
import glob
model.save_pretrained_gguf("gemma-e4b-merv-gguf", tokenizer, quantization_method="q4_k_m")
src = glob.glob("/content/**/gemma-4-e4b*Q4_K_M.gguf", recursive=True)[0]   # text model, not *-mmproj
import os; print("GGUF:", round(os.path.getsize(src)/1e9, 2), "GB", src)

from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get("HF_TOKEN"); repo = "freeideas/merv-gemma4e4b"
api = HfApi(); api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)